# Opening Distribution in the FIDE Candidates Tournament

Comparing opening choices across the 2022, 2024, and 2026 Candidates Tournaments using data from Lichess broadcasts.

In [1]:
import io
import time
import chess.pgn
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Round IDs for each Candidates tournament (Open section), scraped from Lichess
TOURNAMENTS = {
    "2022": {
        "name": "FIDE Candidates Tournament 2022",
        "round_ids": [
            "LsFeKWZU", "sylFQGas", "oe2udItH", "0QuWnLkU", "1ZAF8srK",
            "yF4JxPcn", "OFBhwamI", "fsvj5GFW", "pk9gRSMB", "FWJYzJJJ",
            "16iH8elQ", "ZfmMC2Gh", "oIi8sTms", "ZA07lchF",
        ],
    },
    "2024": {
        "name": "FIDE Candidates 2024 (Open)",
        "round_ids": [
            "AjqSsU1w", "GenKIJ8A", "xQgaUu2y", "CPS9dENa", "MDiLWQ5M",
            "kPqEUBZJ", "X9LGGQoT", "nUycmG6L", "A7SWsX0x", "vfqUR38R",
            "46ohJ8Qt", "eJghIBZe", "Dc0DQMaQ", "S4zisI6M",
        ],
    },
    "2026": {
        "name": "FIDE Candidates 2026 (Open)",
        "round_ids": [
            "uLCZwqAK", "FRTlzP2X", "SDizieNR", "97di6JjX", "liBI9Brw",
            "WmbQrSNz", "aVbIuZ7Q", "XYRPyV8x", "hTVc2Qgj", "G3oSxPgs",
            "seMTFSPr", "9SHysZuu", "rFG1W5Tp", "oBkeHnpi",
        ],
    },
}

In [3]:
def fetch_round_pgn(round_id: str) -> str:
    """Fetch PGN text for a single broadcast round from Lichess."""
    url = f"https://lichess.org/api/broadcast/round/{round_id}.pgn"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.text


def parse_games(pgn_text: str) -> list[dict]:
    """Parse all games from a PGN string and extract header info."""
    games = []
    pgn = io.StringIO(pgn_text)
    while True:
        game = chess.pgn.read_game(pgn)
        if game is None:
            break
        h = game.headers
        games.append({
            "white": h.get("White", ""),
            "black": h.get("Black", ""),
            "result": h.get("Result", ""),
            "eco": h.get("ECO", ""),
            "opening": h.get("Opening", ""),
            "date": h.get("Date", ""),
            "round": h.get("Round", ""),
        })
    return games

In [4]:
all_games = []

for year, info in TOURNAMENTS.items():
    print(f"Fetching {info['name']}...")
    for i, rid in enumerate(info["round_ids"], 1):
        pgn_text = fetch_round_pgn(rid)
        games = parse_games(pgn_text)
        for g in games:
            g["tournament"] = year
        all_games.extend(games)
        print(f"  Round {i}: {len(games)} games")
        time.sleep(0.5)  # be polite to the Lichess API

df = pd.DataFrame(all_games)
# Drop unfinished/unannotated games (Result "*", Opening "?")
df = df[df["result"] != "*"].reset_index(drop=True)
print(f"\nTotal games: {len(df)}")
df.head()

Fetching FIDE Candidates Tournament 2022...


  Round 1: 4 games


  Round 2: 4 games


  Round 3: 4 games


  Round 4: 4 games


...

  Round 12: 4 games


  Round 13: 4 games


  Round 14: 4 games



Total games: 164


,white,black,result,eco,opening,date,round,tournament
0,"Duda, Jan-Krzysztof","Rapport, Richard",1/2-1/2,B44,Sicilian Defense: Taimanov Variation,2022.06.17,1.1,2022
1,"Ding, Liren","Nepomniachtchi, Ian",0-1,A20,English Opening: King's English Variation,2022.06.17,1.2,2022
2,"Caruana, Fabiano","Nakamura, Hikaru",1-0,C65,"Ruy Lopez: Berlin Defense, Anti-Berlin Variation",2022.06.17,1.3,2022
3,"Radjabov, Teimour","Firouzja, Alireza",1/2-1/2,D24,Queen's Gambit Accepted,2022.06.17,1.4,2022
4,Richard Rapport,Alireza Firouzja,1/2-1/2,B53,Sicilian Defense: Chekhover Variation,2022.06.17,?,2022


In [5]:
# ECO codes use a letter (A-E) plus two digits.
# The letter gives the broad family:
#   A = Flank openings, Dutch, Benoni, misc
#   B = Semi-open (Sicilian, Caro-Kann, Pirc, etc.)
#   C = Open games & French (1.e4 e5, French)
#   D = Closed/Semi-closed (Queen's Gambit, Slav, Grunfeld, etc.)
#   E = Indian defenses (Nimzo, Queen's Indian, King's Indian, Catalan, etc.)

df["eco_letter"] = df["eco"].str[0]
df["opening_family"] = df["opening"].str.split(":").str[0].str.split(",").str[0].str.strip()

# Tarrasch Defense is a QGD variation (1.d4 d5 2.c4 e6 3.Nc3 c5)
df["opening_family"] = df["opening_family"].replace("Tarrasch Defense", "Queen's Gambit Declined")

print("Games per tournament:")
print(df.groupby("tournament").size())
print()
print("Sample of parsed fields:")
df[["tournament", "eco", "eco_letter", "opening", "opening_family"]].sample(10, random_state=42)

Games per tournament:
tournament
2022    56
2024    56
2026    52
dtype: int64

Sample of parsed fields:


,tournament,eco,eco_letter,opening,opening_family
135,2026,C24,C,Bishop's Opening: Vienna Hybrid,Bishop's Opening
115,2026,D37,D,Queen's Gambit Declined: Three Knights Variation,Queen's Gambit Declined
131,2026,E05,E,"Catalan Opening: Open Defense, Classical Line",Catalan Opening
55,2022,C43,C,Petrov's Defense: Modern Attack,Petrov's Defense
95,2024,B90,B,"Sicilian Defense: Najdorf Variation, Freak Attack",Sicilian Defense
29,2022,C47,C,"Four Knights Game: Scotch Variation Accepted, ...",Four Knights Game
157,2026,D37,D,Queen's Gambit Declined: Barmen Variation,Queen's Gambit Declined
51,2022,E04,E,Catalan Opening: Open Defense,Catalan Opening
101,2024,C01,C,French Defense: Exchange Variation,French Defense
145,2026,C54,C,"Italian Game: Classical Variation, Giuoco Pian...",Italian Game


## ECO Category Distribution by Year

In [6]:
eco_labels = {
    "A": "A – Flank / Misc",
    "B": "B – Semi-Open (Sicilian, Caro-Kann…)",
    "C": "C – Open / French (1.e4 e5, French…)",
    "D": "D – Closed (QGD, Slav, Grünfeld…)",
    "E": "E – Indian (Nimzo, KID, Catalan…)",
}

eco_ct = df.groupby(["tournament", "eco_letter"]).size().unstack(fill_value=0)
eco_pct = eco_ct.div(eco_ct.sum(axis=1), axis=0) * 100
eco_pct = eco_pct.reindex(columns=sorted(eco_pct.columns))

# Reshape for plotly
eco_plot = eco_pct.reset_index().melt(id_vars="tournament", var_name="eco_letter", value_name="pct")
eco_plot["label"] = eco_plot["eco_letter"].map(eco_labels).fillna(eco_plot["eco_letter"])

fig = px.bar(
    eco_plot,
    x="tournament",
    y="pct",
    color="label",
    barmode="group",
    title="ECO Category Distribution by Candidates Tournament",
    labels={"pct": "% of games", "tournament": "", "label": "ECO Category"},
)
fig.update_layout(yaxis_ticksuffix="%", legend_title_text="")
fig.show()

## Most Popular Openings (All Tournaments Combined)

In [7]:
top_openings = df["opening_family"].value_counts().head(15)

fig = px.bar(
    x=top_openings.values,
    y=top_openings.index,
    orientation="h",
    title="Top 15 Opening Families Across All Three Candidates",
    labels={"x": "Number of games", "y": ""},
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

## Top Openings by Tournament

In [8]:
top_n = 10
top_families = df["opening_family"].value_counts().head(top_n).index

subset = df[df["opening_family"].isin(top_families)]
ct = subset.groupby(["tournament", "opening_family"]).size().unstack(fill_value=0)
ct = ct[top_families]  # preserve popularity order

# Reshape for plotly
ct_plot = ct.reset_index().melt(id_vars="tournament", var_name="opening_family", value_name="count")

fig = px.bar(
    ct_plot,
    x="tournament",
    y="count",
    color="opening_family",
    barmode="group",
    title=f"Top {top_n} Opening Families by Tournament",
    labels={"count": "Number of games", "tournament": "", "opening_family": "Opening"},
)
fig.update_layout(legend_title_text="")
fig.show()

## Opening Diversity: Unique Openings per Tournament

In [9]:
diversity = df.groupby("tournament").agg(
    total_games=("eco", "size"),
    unique_eco_codes=("eco", "nunique"),
    unique_opening_families=("opening_family", "nunique"),
    unique_openings=("opening", "nunique"),
)
diversity

,total_games,unique_eco_codes,unique_opening_families,unique_openings
tournament,,,,
2022,56,33,13,37
2024,56,33,16,38
2026,52,32,13,39


## Full Opening Breakdown Table

Every distinct opening played, with counts per tournament.

In [10]:
full_table = (
    df.groupby(["opening", "eco", "tournament"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .sort_values("total", ascending=False)
)
full_table.head(25)

,tournament,2022,2024,2026,total
opening,eco,,,,
"Ruy Lopez: Berlin Defense, Anti-Berlin Variation",C65,9,5,0,14
"Italian Game: Classical Variation, Giuoco Pianissimo",C54,4,3,1,8
"Petrov's Defense: Classical Attack, Staunton Variation",C42,2,2,0,4
"Ruy Lopez: Morphy Defense, Anderssen Variation",C77,1,3,0,4
Sicilian Defense: Nyezhmetdinov-Rossolimo Attack,B30,0,4,0,4
"Catalan Opening: Open Defense, Classical Line",E05,1,0,3,4
Queen's Gambit Declined: Three Knights Variation,D37,0,0,4,4
"Four Knights Game: Scotch Variation Accepted, Main Line",C47,2,1,0,3
Petrov's Defense: Paulsen Attack,C42,0,1,2,3
